# Tasks 2 & 3: Model Explainability and Business Recommendations

**Objective**: Evaluate and interpret the pre-trained fraud detection models for both E-commerce and Credit Card transactions using performance metrics and SHAP analysis. Provide actionable business recommendations based on the findings.

## 1. Setup and Configurations

In [ ]:
import sys
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import shap
import joblib
from pathlib import Path

# Add src to path for modular imports
sys.path.append('../')

from src.visualisation import plotter as plotter_module
from src.pipeline import tabular_modeling as tm

# Configuration
pd.set_option("display.max_columns", 120)
pd.options.display.float_format = "{:.4f}".format

ROOT = Path("..").resolve()
FIGURES_DIR = ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Initialize Plotter
plotter = plotter_module.Plotter(figures_dir=FIGURES_DIR)

---

## 2. E-commerce Fraud Model Analysis

### 2.1 Load Data and Model

In [ ]:
FRAUD_DATA_PATH = ROOT / "data" / "processed" / "Fraud_Data_Processed.csv"
FRAUD_MODEL_PATH = ROOT / "models" / "fraud.pkl"

# Load Data
fraud_df = pd.read_csv(FRAUD_DATA_PATH)
print(f"E-commerce Data Shape: {fraud_df.shape}")

# Pre-split separation (replicating training logic to get test set)
TARGET_COL = "class"
DROP_COLS = ["user_id", "signup_time", "purchase_time", "device_id", "ip_address"]

X_train, X_test, y_train, y_test = tm.split_features_target(
    fraud_df, target=TARGET_COL, drop_cols=DROP_COLS, stratify=True, test_size=0.2, random_state=42
)

# Load Model
fraud_pipeline = joblib.load(FRAUD_MODEL_PATH)
print("E-commerce Model loaded successfully.")

### 2.2 Performance Evaluation

In [ ]:
print("Performance Metrics:")
metrics = tm.evaluate_classification({"E-commerce Model": fraud_pipeline}, X_test, y_test)
display(metrics)

y_pred = fraud_pipeline.predict(X_test)
y_proba = fraud_pipeline.predict_proba(X_test)[:, 1]

plotter.plot_confusion_matrix(y_test, y_pred, title="Confusion Matrix: E-commerce Fraud")
plotter.plot_roc_curve(y_test, y_proba, title="ROC Curve: E-commerce Fraud")

### 2.3 Feature Importance (Built-in)
We visualize the top 10 most important features according to the model's internal gain/gini calculation.

In [ ]:
classifier = fraud_pipeline.named_steps['model']
prep = fraud_pipeline.named_steps['prep']

if hasattr(classifier, 'feature_importances_'):
    importances = classifier.feature_importances_
    # Recover feature names after preprocessing
    num_cols = list(prep.named_transformers_['num'].get_feature_names_out())
    # Handle categorical names (usually cat__source_Ads etc)
    # Note: tabular_modeling uses ('cat', Pipeline(steps=[(..., encoder)]), list(cols))
    cat_pipeline = prep.named_transformers_['cat']
    ohe = cat_pipeline.named_steps['encoder']
    cat_orig_cols = prep.transformers_[1][2] # list of categorical columns passed
    ohe_cols = list(ohe.get_feature_names_out(cat_orig_cols))
    feature_names = num_cols + ohe_cols
    
    feat_imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
    feat_imp_df = feat_imp_df.sort_values('importance', ascending=False).head(10)
    
    plotter.plot_bar(feat_imp_df, x='feature', y='importance', title="Top 10 Built-in Feature Importances (E-commerce)")

### 2.4 SHAP Global Analysis

In [ ]:
# Preprocess test set for SHAP
X_test_transformed = prep.transform(X_test)
if hasattr(X_test_transformed, "toarray"): X_test_transformed = X_test_transformed.toarray()
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)

# Calculate SHAP values (subset for speed)
explainer = shap.TreeExplainer(classifier)
shap_values = explainer.shap_values(X_test_df.sample(1000, random_state=42))

# Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_df.sample(1000, random_state=42), show=False)
plt.title("SHAP Summary Plot (E-commerce Fraud)")
plt.tight_layout()
plotter._finalize("SHAP Summary Plot E-commerce", "SHAP value", "Features")

### 2.5 SHAP Local Analysis (Force Plots)
Explaining specific categories: True Positive, False Positive, False Negative.

In [ ]:
def explain_instance(indices, case_label):
    if len(indices) > 0:
        idx = indices[0]
        print(f"--- {case_label} (Index: {idx}) ---")
        inst = X_test_df.iloc[[idx]]
        sv = explainer.shap_values(inst)
        shap.initjs()
        exp_val = explainer.expected_value
        if isinstance(exp_val, (list, np.ndarray)) and len(exp_val) > 1: exp_val = exp_val[1]
        display(shap.force_plot(exp_val, sv[0] if isinstance(sv, list) else sv, inst))
    else:
        print(f"No instances found for {case_label}")

tp_idx = np.where((y_test.values == 1) & (y_pred == 1))[0]
fp_idx = np.where((y_test.values == 0) & (y_pred == 1))[0]
fn_idx = np.where((y_test.values == 1) & (y_pred == 0))[0]

explain_instance(tp_idx, "True Positive (Correct Fraud Identification)")
explain_instance(fp_idx, "False Positive (Legitimate Flagged as Fraud)")
explain_instance(fn_idx, "False Negative (Fraud Missed by Model)")

---

## 3. Credit Card Fraud Model Analysis

### 3.1 Load Data and Model

In [ ]:
CC_DATA_PATH = ROOT / "data" / "raw" / "creditcard.csv"
CC_MODEL_PATH = ROOT / "models" / "creaditcard.pkl"

# Load Data
cc_df = pd.read_csv(CC_DATA_PATH)
print(f"Credit Card Data Shape: {cc_df.shape}")

# Split (replicating train_creditcard_models.py logic)
X_train_cc, X_test_cc, y_train_cc, y_test_cc = tm.split_features_target(
    cc_df, target="Class", stratify=True, test_size=0.2, random_state=42
)

# Load Model
cc_pipeline = joblib.load(CC_MODEL_PATH)
print("Credit Card Model loaded successfully.")

### 3.2 Performance Evaluation

In [ ]:
print("Performance Metrics:")
metrics_cc = tm.evaluate_classification({"Credit Card Model": cc_pipeline}, X_test_cc, y_test_cc)
display(metrics_cc)

y_pred_cc = cc_pipeline.predict(X_test_cc)
y_proba_cc = cc_pipeline.predict_proba(X_test_cc)[:, 1]

plotter.plot_confusion_matrix(y_test_cc, y_pred_cc, title="Confusion Matrix: Credit Card Fraud")
plotter.plot_roc_curve(y_test_cc, y_proba_cc, title="ROC Curve: Credit Card Fraud")

### 3.3 Feature Importance (Built-in)

In [ ]:
classifier_cc = cc_pipeline.named_steps['model']
prep_cc = cc_pipeline.named_steps['prep']

if hasattr(classifier_cc, 'feature_importances_'):
    feature_names_cc = list(prep_cc.get_feature_names_out())
    feat_imp_df_cc = pd.DataFrame({'feature': feature_names_cc, 'importance': classifier_cc.feature_importances_})
    feat_imp_df_cc = feat_imp_df_cc.sort_values('importance', ascending=False).head(10)
    plotter.plot_bar(feat_imp_df_cc, x='feature', y='importance', title="Top 10 Feature Importances (Credit Card)")

### 3.4 SHAP Global Analysis

In [ ]:
X_test_transformed_cc = prep_cc.transform(X_test_cc)
X_test_df_cc = pd.DataFrame(X_test_transformed_cc, columns=feature_names_cc)

explainer_cc = shap.TreeExplainer(classifier_cc)
shap_values_cc = explainer_cc.shap_values(X_test_df_cc.sample(1000, random_state=42))

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_cc, X_test_df_cc.sample(1000, random_state=42), show=False)
plt.title("SHAP Summary Plot (Credit Card Fraud)")
plt.tight_layout()
plotter._finalize("SHAP Summary Plot Credit Card", "SHAP value", "Features")

### 3.5 SHAP Local Analysis (Force Plots)

In [ ]:
tp_idx_cc = np.where((y_test_cc.values == 1) & (y_pred_cc == 1))[0]
fp_idx_cc = np.where((y_test_cc.values == 0) & (y_pred_cc == 1))[0]
fn_idx_cc = np.where((y_test_cc.values == 1) & (y_pred_cc == 0))[0]

# Re-using instance explainer logic for CC
def explain_cc_instance(indices, label):
    if len(indices) > 0:
        idx = indices[0]
        inst = X_test_df_cc.iloc[[idx]]
        sv = explainer_cc.shap_values(inst)
        shap.initjs()
        exp_val = explainer_cc.expected_value
        if isinstance(exp_val, (list, np.ndarray)) and len(exp_val) > 1: exp_val = exp_val[1]
        display(shap.force_plot(exp_val, sv[0] if isinstance(sv, list) else sv, inst))
    else: print(f"No instances for {label}")

explain_cc_instance(tp_idx_cc, "Credit Card TP")
explain_cc_instance(fp_idx_cc, "Credit Card FP")
explain_cc_instance(fn_idx_cc, "Credit Card FN")

---

## 4. Interpretation and Recommendations

### 4.1 Interpretation
1. **Top Drivers**: Features like `time_since_signup` (E-commerce) and `V17`, `V14` (Credit Card) are primary drivers of fraud scores.
2. **SHAP vs Built-in**: Consistent top features, but SHAP provides magnitude and direction of the effect.
3. **Insights**: 
   - E-commerce: Extremely low `time_since_signup` is highly predictive of fraud.
   - Credit Card: Specific latent features (likely transaction patterns) show sharp impact.

### 4.2 Business Recommendations
1. **Instant Action for New Signups**: Transactions made within seconds of account creation should be automatically blocked or put in high-friction queue (Step-up Auth).
2. **Geo-Velocity Validation**: If `ip_txn_count` across different devices is high for the same IP, trigger immediate device-fingerprinting check.
3. **High-Value purchase limit**: First-time purchases that exceed the median `purchase_value` should require 3DS verification.